# 02 - Data: the step that decides everything

Three engineering points the handout calls the easy-to-get-wrong ones:

1. **Chat format** - the gold JSON goes in the *assistant* turn, never the user
   turn, so the loss mask trains on the answer only.
2. **EOS** - the tokenizer chat template appends the stop token; we do not
   hand-roll it.
3. **Decontamination** - split first, then drop any train item that matches a
   test item. Nothing trained on may appear in what it is graded on.

In [ ]:
import sys; sys.path.insert(0, '../src')
from qlora_lab import schema, synth, dataset, predict, evaluate, train, serve, agent

In [ ]:
examples = synth.gen(800, seed=7)
parts = dataset.split(examples, n_test=100, n_val=100)
train, removed = dataset.decontaminate(parts['train'], parts['test'])
print('decontam removed', removed, 'leaving', len(train), 'train')

### What one SFT record looks like
System + user + assistant; assistant is the gold JSON.

In [ ]:
rec = dataset.to_chat(train[0])
import json; print(json.dumps(rec, indent=2)[:600])

### Write the files TRL will read

In [ ]:
dataset.write_jsonl([dataset.to_chat(e) for e in train], '../data/train.jsonl')
dataset.write_jsonl([dataset.to_chat(e) for e in parts['val']], '../data/val.jsonl')
dataset.write_jsonl(parts['test'], '../data/test.jsonl')
print('wrote train/val/test')

For real data, replace `synth.gen` with your loader: de-identified production
logs, or a strong model generating then a human spot-checking. Everything
downstream is unchanged.